# 00 — Fresh Run Initialization

Start here to wipe all pipeline artifacts and run the notebooks cleanly from scratch.

Each notebook in the pipeline writes to a specific data directory. Running this
notebook clears them all so you can re-run 01 → 11 without stale data interfering.

| Directory | Produced by |
|-----------|-------------|
| `01_corpus/` | NB01 — Corpus Collection |
| `02_knowledge/` | NB02–07b — Knowledge extraction + accumulated pairs |
| `03_raw_generated/` | NB08 — Synthetic data generation |
| `04_validated/` | NB09 — Validation |
| `05_dataset/` | NB10 — Dataset assembly |
| `adapters/` | NB04 — Warm-start LoRA adapters |
| `wiki/` | NB07b — Wiki clone |
| `sft_data/` | NB08 — RL fine-tune data |
| `../models/` | NB11 — Fine-tuned model checkpoints |

The base model cache is **not** touched — no re-downloading needed.

In [1]:
import sys, importlib, shutil
from pathlib import Path

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

# Every directory produced by the pipeline, paired with a description.
# Order matches pipeline stage order so the output reads top-to-bottom.
STAGE_DIRS = [
    (DATA_ROOT / '01_corpus',        'NB01  — Corpus Collection'),
    (DATA_ROOT / '02_knowledge',     'NB02–07b — Knowledge + Accumulated Pairs'),
    (DATA_ROOT / '03_raw_generated', 'NB08  — Synthetic Data Generation'),
    (DATA_ROOT / '04_validated',     'NB09  — Validation'),
    (DATA_ROOT / '05_dataset',       'NB10  — Dataset Assembly'),
    (DATA_ROOT / 'adapters',         'NB04  — Warm-start LoRA Adapters'),
    (DATA_ROOT / 'wiki',             'NB07b — Wiki Clone'),
    (DATA_ROOT / 'sft_data',         'NB08  — RL Fine-tune Data'),
    (SCRIPT_DIR / '../models',       'NB11  — Fine-tuned Model Checkpoints'),
]

print('Directories that will be cleared:\n')
total_files = 0
for d, label in STAGE_DIRS:
    p = Path(d)
    if p.exists():
        n = sum(1 for _ in p.rglob('*') if _.is_file())
        total_files += n
        print(f'  [{label}]')
        print(f'    {p}  ({n} files)')
    else:
        print(f'  [{label}]')
        print(f'    {p}  — not present, will be skipped')

print(f'\nTotal files to remove: {total_files}')
print('\nRun the next cell to confirm and delete.')

Directories that will be cleared:

  [NB01  — Corpus Collection]
    /Volumes/Models/data/../data/01_corpus  — not present, will be skipped
  [NB02–07b — Knowledge + Accumulated Pairs]
    /Volumes/Models/data/../data/02_knowledge  (3 files)
  [NB08  — Synthetic Data Generation]
    /Volumes/Models/data/../data/03_raw_generated  — not present, will be skipped
  [NB09  — Validation]
    /Volumes/Models/data/../data/04_validated  — not present, will be skipped
  [NB10  — Dataset Assembly]
    /Volumes/Models/data/../data/05_dataset  — not present, will be skipped
  [NB04  — Warm-start LoRA Adapters]
    /Volumes/Models/data/../data/adapters  (7 files)
  [NB07b — Wiki Clone]
    /Volumes/Models/data/../data/wiki  — not present, will be skipped
  [NB08  — RL Fine-tune Data]
    /Volumes/Models/data/../data/sft_data  — not present, will be skipped
  [NB11  — Fine-tuned Model Checkpoints]
    /Users/kris/Projects/ARO-App/Train/script/../models  — not present, will be skipped

Total files to 

## Confirm and Delete

**This is irreversible.** Review the file counts above, then run the cell below.

In [ ]:
cleared, skipped = [], []
_file_counts = {}   # label → files removed (for graph)

for d, label in STAGE_DIRS:
    p = Path(d)
    if p.exists():
        n = sum(1 for _ in p.rglob('*') if _.is_file())
        shutil.rmtree(p)
        p.mkdir(parents=True, exist_ok=True)
        cleared.append(label)
        _file_counts[label] = n
    else:
        skipped.append(label)
        _file_counts[label] = 0

# Recreate the ADAPTER_DIR that config.py mkdir's on import
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print(f'\nDone.  {len(cleared)} cleared,  {len(skipped)} skipped.')
print('\nNext: run notebooks 01 → 11 in order to regenerate all data.')

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '00_init.png'

_labels  = [label for _, label in STAGE_DIRS]
_counts  = [_file_counts.get(label, 0) for label in _labels]
_colors  = ['#2ecc71' if label in cleared else '#bdc3c7' for label in _labels]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(_labels, _counts, color=_colors, edgecolor='white', height=0.65)
ax.set_xlabel('Files removed')
ax.set_title(
    f'Fresh Run — {len(cleared)} stages cleared  ·  {sum(_counts):,} files removed',
    fontsize=13, fontweight='bold'
)
for bar, n, label in zip(bars, _counts, _labels):
    tag  = f'{n:,} files' if n else 'was empty'
    xpos = max(n + max(_counts) * 0.01, max(_counts) * 0.015)
    ax.text(xpos, bar.get_y() + bar.get_height() / 2,
            tag, va='center', fontsize=9, color='#555')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
ax.set_facecolor('#f9f9f9')
fig.patch.set_facecolor('#fafafa')
fig.tight_layout()
fig.savefig(_out, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved: {_out}')